In [1]:
cfg= {
    "vocab_size": 1000,
    "hidden_size": 256,
    
    "num_hidden_layers": 6,
    "num_attention_heads": 8,
    "num_key_value_heads": 8,

    "intermediate_size": 1024,

    "max_position_embeddings": 128,

    "eps": 1e-6,
    "rope_theta": 10000.0,

    "dropout": 0.0,
    "bias": False,
}

In [2]:
import torch
from BPE import BPE
from data_loader import load_dataset_,create_loader

ds=load_dataset_()
corpus = "\n".join(sample["text"] for sample in ds)
print(len(corpus))
tokenizer=BPE(vocab_size=1000)
tokenizer.train(corpus)
dataloader=create_loader(corpus,tokenizer,max_length=128,stride=128,shuffle=False,batch_size=8)

data_iter=iter(dataloader)
ip,tgt=next(data_iter)
print("\nInputs\n",ip)
print("\ntarget\n",tgt)

print("\nInput shape:\n",ip.shape)
print("\nTarget shape:\n",tgt.shape)

token_ids=tokenizer.encode(corpus)
torch.save(token_ids,"token_ids.pt")

5905283

Inputs
 tensor([[701, 687, 620,  ..., 428, 497, 417],
        [847, 595, 834,  ..., 538, 801, 482],
        [782, 593,  22,  ..., 482, 531, 748],
        ...,
        [333, 573, 469,  ..., 542, 529, 855],
        [828,   1, 608,  ..., 773, 862, 417],
        [707, 333, 351,  ..., 608, 487,  36]])

target
 tensor([[687, 620, 608,  ..., 497, 417, 847],
        [595, 834, 743,  ..., 801, 482, 782],
        [593,  22, 588,  ..., 531, 748, 872],
        ...,
        [573, 469, 525,  ..., 529, 855, 828],
        [  1, 608, 487,  ..., 862, 417, 707],
        [333, 351, 426,  ..., 487,  36, 619]])

Input shape:
 torch.Size([8, 128])

Target shape:
 torch.Size([8, 128])


In [3]:
from Embedding import Embedding
embedding_layer=Embedding(cfg)
emb=embedding_layer(ip)
from RMSNorm import RMSNorm
rms=RMSNorm(cfg)
print(emb.shape)
print(rms(emb).shape)

torch.Size([8, 128, 256])
torch.Size([8, 128, 256])


In [4]:
from ROPE import ROPE
rope = ROPE(cfg)

x = torch.randn(
    2,
    128,
    cfg["num_attention_heads"],
    cfg["hidden_size"] // cfg["num_attention_heads"]
)

out = rope(x)

print(x.shape)
print(out.shape)

torch.Size([2, 128, 8, 32])
torch.Size([2, 128, 8, 32])


In [5]:
import torch
import torch.nn as nn
from MHA import MHA


In [6]:
norma=rms(emb)
mha=MHA(cfg)
mha(norma).shape

torch.Size([8, 128, 256])

In [9]:
from SwiGLU import SwiGLU
ffn = SwiGLU(cfg)

x = torch.randn(8, 128, cfg["hidden_size"])

out = ffn(x)

print(out.shape)

torch.Size([8, 128, 256])
